# Data inventory and coverage audit

Check observed country-industry-month coverage. Source manifest and classifications are separately verified by Codex.

**Real data only.** Run ingestion first. Never interpret an empty or skipped model as a finding.

In [ ]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, load_panel, load_cases, output

In [ ]:
import pandas as pd
p=load_panel()
target_countries=['IND','USA','DEU']; target_industries=['10','13','20','21','22','24','25','27','28','29']
coverage=p.dropna(subset=['production_yoy_pct']).groupby(['country','industry_code']).agg(
    n_months=('date','nunique'),first=('date','min'),last=('date','max')).reset_index()
full=pd.MultiIndex.from_product([target_countries,target_industries],names=['country','industry_code']).to_frame(index=False)
coverage=full.merge(coverage,how='left',on=['country','industry_code'])
coverage['status']=coverage['n_months'].apply(lambda x:'unavailable' if pd.isna(x) or x==0 else 'observed')
display(coverage)
output('coverage_audit',{'status':'audited','target_panels':30,'target_event_rows':30*len(__import__('common').event_config()),'observed_panels':int((coverage.status=='observed').sum()),
'panels':coverage.assign(first=coverage['first'].astype(str),last=coverage['last'].astype(str)).fillna({'n_months':0}).where(pd.notna(coverage),None).to_dict('records')})